In [1]:
import numpy as np
import math
import random

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ============================================================
# 1. INPUT DATA
# ============================================================

X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.7786275 ],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.9265642],
    [0.926564, 1.026564],
    [0.747504, 0.20897],
    [0.683406, 0.063769],
    [0.583137, 0.012549],
    [0.991457, 0.001744],
    [0.991457, 0.001744]
])

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156
])

# Clip to [0,1]
X = np.clip(X, 0, 1)

# ============================================================
# 2. TRAIN-ONLY NOISE (Dropout-like regularisation)
# ============================================================

class TrainOnlyNoise:
    """
    Adds Gaussian noise ONLY during training (fit_transform),
    but NOT during inference (transform).
    """
    def __init__(self, sigma=0.01, seed=0):
        self.sigma = sigma
        self.seed = seed

    def fit(self, X, y=None):
        return self

    def fit_transform(self, X, y=None):
        rng = np.random.default_rng(self.seed)
        return X + rng.normal(0, self.sigma, size=X.shape)

    def transform(self, X):
        return X


def make_model(seed, hidden=(64, 32), alpha=5e-5, lr=0.03,
               sigma=0.01, max_iter=3000, tol=1e-7, n_iter_no_change=30):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("noise", TrainOnlyNoise(sigma=sigma, seed=seed)),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=hidden,
            activation="relu",
            solver="adam",
            learning_rate="adaptive",
            learning_rate_init=lr,
            alpha=alpha,
            early_stopping=True,
            n_iter_no_change=n_iter_no_change,
            max_iter=max_iter,
            tol=tol,
            random_state=seed
        ))
    ])

# ============================================================
# 3. RANDOM SEARCH: Hyperparameter tuning via CV
# ============================================================

def cv_mse_for_config(X, y, cfg, n_splits=4):
    """
    Small-data CV:
    - KFold with shuffle
    - Ensemble averaging inside each fold improves stability
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    fold_mses = []

    for tr_idx, te_idx in kf.split(X):
        Xtr, Xte = X[tr_idx], X[te_idx]
        ytr, yte = y[tr_idx], y[te_idx]

        preds = []
        for i in range(cfg["ens_cv"]):
            m = make_model(
                seed=1000 + i,
                hidden=cfg["hidden"],
                alpha=cfg["alpha"],
                lr=cfg["lr"],
                sigma=cfg["sigma"],
                max_iter=cfg["max_iter"],
                tol=cfg["tol"],
                n_iter_no_change=cfg["ninc"]
            )
            m.fit(Xtr, ytr)
            preds.append(m.predict(Xte))

        pred_mean = np.mean(np.vstack(preds), axis=0)
        fold_mses.append(mean_squared_error(yte, pred_mean))

    return float(np.mean(fold_mses))


def random_search_best_config(X, y, n_trials=40, seed=7):
    random.seed(seed)

    hidden_choices = [(32, 16), (64, 32), (64, 32, 16), (128, 64, 32)]
    alpha_choices  = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3]
    lr_choices     = [1e-3, 3e-3, 1e-2, 3e-2]
    sigma_choices  = [0.0, 0.005, 0.01, 0.015, 0.02]
    tol_choices    = [1e-6, 1e-7]
    ninc_choices   = [20, 30, 40]
    max_iter_choices = [3000, 5000]
    ens_cv_choices = [3, 5]

    best_cfg = None
    best_mse = float("inf")

    for _ in range(n_trials):
        cfg = {
            "hidden": random.choice(hidden_choices),
            "alpha":  random.choice(alpha_choices),
            "lr":     random.choice(lr_choices),
            "sigma":  random.choice(sigma_choices),
            "tol":    random.choice(tol_choices),
            "ninc":   random.choice(ninc_choices),
            "max_iter": random.choice(max_iter_choices),
            "ens_cv": random.choice(ens_cv_choices),
        }

        mse = cv_mse_for_config(X, y, cfg)
        if mse < best_mse:
            best_mse = mse
            best_cfg = cfg

    return best_cfg, best_mse


best_cfg, best_cv_mse = random_search_best_config(X, y, n_trials=40, seed=7)

# ============================================================
# 4. FIT FINAL ENSEMBLE USING BEST CONFIG
# ============================================================

def fit_ensemble(X, y, cfg, n=16, seed0=300):
    models = []
    for i in range(n):
        m = make_model(
            seed=seed0 + i,
            hidden=cfg["hidden"],
            alpha=cfg["alpha"],
            lr=cfg["lr"],
            sigma=cfg["sigma"],
            max_iter=cfg["max_iter"],
            tol=cfg["tol"],
            n_iter_no_change=cfg["ninc"]
        )
        m.fit(X, y)
        models.append(m)
    return models


def ensemble_predict(models, Xcand):
    preds = np.vstack([m.predict(Xcand) for m in models])
    mu = preds.mean(axis=0)
    std = preds.std(axis=0, ddof=1) + 1e-9
    return mu, std


# ============================================================
# 5. EI / PI utilities
# ============================================================

def erf_vec(x):
    return np.vectorize(math.erf)(x)

def compute_pi_ei(mu, std, y_best, xi=0.005):
    z = (mu - y_best - xi) / std
    pdf = (1 / np.sqrt(2*np.pi)) * np.exp(-0.5 * z*z)
    cdf = 0.5 * (1 + erf_vec(z / np.sqrt(2)))
    ei = (mu - y_best - xi) * cdf + std * pdf
    ei[std <= 0] = 0
    return cdf, ei


# ============================================================
# 6. Candidate generation (global + local)
# ============================================================

models = fit_ensemble(X, y, best_cfg, n=16)

idx_best = int(np.argmax(y))
x_best = X[idx_best]
y_best = float(y[idx_best])

rng = np.random.default_rng(999)

N_global = 30000
Xcand = rng.uniform(0, 1, size=(N_global, 2))

# Local exploration around current best (tuned slightly tighter vs Week 6)
local_scale = 0.035
local_n = 8000
local = rng.normal(loc=x_best, scale=local_scale, size=(local_n, 2))
local = np.clip(local, 0, 1)

Xcand = np.vstack([Xcand, local])

# ============================================================
# 7. Acquisition: EI + "not-duplicate" + mild exploration (UCB filter)
# ============================================================

mu, std = ensemble_predict(models, Xcand)
pi, ei = compute_pi_ei(mu, std, y_best=y_best, xi=0.005)

# Distance from existing points to avoid repeats / near repeats
dists = np.sqrt(((Xcand[:, None, :] - X[None, :, :])**2).sum(axis=2))
min_dist = dists.min(axis=1)

# Hybrid selection:
# - Use UCB to avoid EI collapsing when y_best is very high
# - Enforce min distance so we don't re-sample the same point
beta = 2.5
ucb = mu + beta * std

thr = 0.01  # minimum separation from existing points
order = np.argsort(-ucb)

chosen = None
for k in order:
    if min_dist[k] >= thr:
        chosen = int(k)
        break

x_next = Xcand[chosen]
mu_next = float(mu[chosen])
std_next = float(std[chosen])
pi_next = float(pi[chosen])
ei_next = float(ei[chosen])

# ============================================================
# 8. OUTPUT
# ============================================================

print("================================================")
print("WEEK 7 FUNCTION 2 — HYPERPARAMETER TUNING RESULT")
print("================================================")
print(f"Best CV-MSE (lower is better): {best_cv_mse:.6f}")
print("Best tuned hyperparameters:")
for k, v in best_cfg.items():
    print(f"  {k}: {v}")

print("\n================================================")
print("CURRENT BEST OBSERVED")
print("================================================")
print(f"x_best = {x_best}, y_best = {y_best:.6f}")

print("\n================================================")
print("RECOMMENDED NEXT POINT (Hybrid UCB/EI, non-duplicate)")
print("================================================")
print(f"x_next     = {x_next}")
print(f"μ(x_next)  = {mu_next:.6f}")
print(f"σ(x_next)  = {std_next:.6f}")
print(f"PI         = {pi_next:.4f}")
print(f"EI         = {ei_next:.6f}")
print(f"min_dist   = {float(min_dist[chosen]):.6f} (distance to nearest existing point)")


WEEK 7 FUNCTION 2 — HYPERPARAMETER TUNING RESULT
Best CV-MSE (lower is better): 0.057410
Best tuned hyperparameters:
  hidden: (64, 32)
  alpha: 1e-05
  lr: 0.03
  sigma: 0.0
  tol: 1e-06
  ninc: 20
  max_iter: 3000
  ens_cv: 5

CURRENT BEST OBSERVED
x_best = [0.683406 0.063769], y_best = 0.629731

RECOMMENDED NEXT POINT (Hybrid UCB/EI, non-duplicate)
x_next     = [0.9994772  0.02152806]
μ(x_next)  = 0.249379
σ(x_next)  = 0.218948
PI         = 0.0392
EI         = 0.003454
min_dist   = 0.021348 (distance to nearest existing point)
